[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/Shuhrat-1/MLP_assignments/blob/main/assignment_1.ipynb)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from torch.utils.data import TensorDataset, DataLoader

# Set random seed for reproducibility
torch.manual_seed(42)

url = "https://raw.githubusercontent.com/Shuhrat-1/MLP_assignments/main/winequality-white.csv"
data = pd.read_csv(url, sep=';')

print(data.info())
print(data.describe())


### Data Preprocessing
Splitting the dataset into training and test sets and standardizing the features.

In [ ]:
# Input features (X) and output/response variable (y)
X = data.drop('quality', axis=1).values
y = data['quality'].values.reshape(-1, 1)

# Split into train dataset and independent test dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Split dataset into:
# train dataset (used to train the model)
# independent test dataset (used to evaluate predictions)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train dataset shape:", X_train.shape, y_train.shape)
print("Independent test dataset shape:", X_test.shape, y_test.shape)

### Convert to PyTorch tensors

In [ ]:
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

### Define the neural network model for regression

In [ ]:
class WineQualityModel(nn.Module):
    def __init__(self, input_size, hidden_size=16, output_size=1):
        super(WineQualityModel, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x is the input to the model
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        # output is the predicted wine quality
        return x

# Number of input variables (features)
input_size = X_train.shape[1]

# One output value: predicted quality
output_size = 1

### Model Training

In [ ]:
# Create train dataset and dataloader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# Batch: small subset of the train dataset used to update the model
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Initialize the model
model = WineQualityModel(input_size=input_size, output_size=output_size)

# Define loss function
loss_function = nn.MSELoss()

# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=0.001)

# Training parameters
num_epochs = 100
train_losses = []

# Training loop
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        # Forward pass
        outputs = model(X_batch)
        loss = loss_function(outputs, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

### Prediction step

In [ ]:
# Predict wine quality using the trained model
model.eval()
with torch.no_grad():
    y_pred_tensor = model(X_test_tensor)
    y_pred = y_pred_tensor.numpy()
    y_test_np = y_test_tensor.numpy()
    
    # Calculate testing MSE and RMSE
    test_mse = mean_squared_error(y_test_np, y_pred)
    rmse = test_mse ** 0.5

    print(f"Test MSE: {test_mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")

### Plot actual vs predicted values

In [ ]:
# Plot actual vs predicted values
y_test_flat = y_test_np.flatten()
y_pred_flat = y_pred.flatten()

plt.figure(figsize=(8, 6))
plt.scatter(y_test_flat, y_pred_flat, alpha=0.6)
plt.xlabel("Actual Quality")
plt.ylabel("Predicted Quality")
plt.title("Actual vs Predicted Quality")

min_val = min(y_test_flat.min(), y_pred_flat.min())
max_val = max(y_test_flat.max(), y_pred_flat.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--')

plt.grid(True)
plt.show()

### Plot loss vs epochs

In [ ]:
# Plot loss vs epochs
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_losses)
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Training Loss over Epochs')
plt.grid(True)
plt.savefig('loss_vs_epochs.png')
plt.show()